# Desenvolvimento do modelo preditivo para descobrir os sobreviventes do Titanic

### Imports

In [21]:
import pandas as pd
import sklearn as sk
import numpy as np
from scipy.stats import randint, uniform
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report


## Analisando os datasets

In [22]:
gender_path = "./datas/gender_submission.csv"
test_path = "./datas/test.csv"
train_path = "./datas/train.csv"

In [23]:
data_gender = pd.read_csv(gender_path)
data_test = pd.read_csv(test_path)
data_train = pd.read_csv(train_path)

In [24]:
data_train.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [25]:
data_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [26]:
contagem = data_train["Survived"]
print(contagem.value_counts())
contagem.isnull().sum()

Survived
0    549
1    342
Name: count, dtype: int64


np.int64(0)

## Funções de engenharia de Feature

In [27]:
def eng_feature_cabin(col_cabin):
    if pd.isnull(col_cabin):
        col_cabin = "U"
    return col_cabin

In [28]:
def eng_feature_Embarked(col_embarked):
    if col_embarked == "S":
        col_embarked = 1
    elif col_embarked == "C":
        col_embarked = 2
    elif col_embarked == "Q":
        col_embarked = 3
    return col_embarked

In [29]:
def eng_feature_sex(col_sex):
    if col_sex == "male":
        col_sex = 1
    elif col_sex == "female":
        col_sex = 2
    return col_sex

In [30]:
def eng_feature_type_persona(dataframe):
    if dataframe["Age"] <= 3.0:
        return 0
    elif dataframe["Age"] <= 12.0:  # Não precisa testar se é > 3, pois se chegou aqui é porque falhou no teste de cima
        return 1
    elif dataframe["Age"] <= 18.0:
        return 2
    elif dataframe["Age"] <= 60.0:
        return 3
    else:                     # Se chegou aqui, com certeza é maior que 60
        return 4
    
# Outra forma de fazer a função acima, usando o pd.cut

# 1. Definimos os limites (bins) das idades. Do 0 ao 3, 3 ao 12, etc. (O 150 é o limite máximo)
# limites = [0, 3, 12, 18, 60, 150]

# 2. Definimos os rótulos (labels) que você quer retornar
# rotulos = [0, 1, 2, 3, 4]

# 3. Cortamos a coluna Age usando esses limites
# data_test["type_persona"] = pd.cut(data_test_mod["Age"], bins=limites, labels=rotulos)

In [31]:
data_test_mod = data_test.copy()
data_test_mod["Cabin"] = data_test_mod["Cabin"].apply(eng_feature_cabin)
data_test_mod["Embarked"] = data_test_mod["Embarked"].apply(eng_feature_Embarked)
data_test_mod["Sex"] = data_test_mod["Sex"].apply(eng_feature_sex)
data_test_mod["type_persona"] = data_test_mod.apply(eng_feature_type_persona, axis=1)

In [32]:
data_train_mod = data_train.copy()
data_train_mod["Cabin"] = data_train_mod["Cabin"].apply(eng_feature_cabin)
data_train_mod["Embarked"] = data_train_mod["Embarked"].apply(eng_feature_Embarked)
data_train_mod["Sex"] = data_train_mod["Sex"].apply(eng_feature_sex)
data_train_mod["type_persona"] = data_train_mod.apply(eng_feature_type_persona, axis=1)

In [33]:
contagem = data_test_mod["type_persona"]
print(contagem.value_counts())
contagem.isnull().sum()

type_persona
3    267
4     97
2     29
1     14
0     11
Name: count, dtype: int64


np.int64(0)

In [34]:
train = data_train_mod.drop(["PassengerId", "Name", "Ticket", "Cabin"], axis=1)
test = data_test_mod.drop(["PassengerId", "Name", "Ticket", "Cabin"], axis=1)

In [35]:
test.info()

<class 'pandas.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Pclass        418 non-null    int64  
 1   Sex           418 non-null    int64  
 2   Age           332 non-null    float64
 3   SibSp         418 non-null    int64  
 4   Parch         418 non-null    int64  
 5   Fare          417 non-null    float64
 6   Embarked      418 non-null    int64  
 7   type_persona  418 non-null    int64  
dtypes: float64(2), int64(6)
memory usage: 26.3 KB


## Construção do modelo tipo Random Forest Classifier

In [36]:
y = train["Survived"]
X = train.drop(["Survived"], axis=1)

In [37]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [38]:
param_dist = {
    'n_estimators': randint(100, 1100),
    'max_features': ['sqrt', 'log2', 1.0],
    'max_depth': randint(4, 12),
    'min_samples_split': randint(2, 8),
    'min_samples_leaf': randint(1, 8),
    'max_samples': [0.6, 0.7, 0.8, 0.9, 1.0], # Determina o tamanho das amostras de dados em que cada árvore recebe
    'ccp_alpha': uniform(0, 0.02) # Efetua uma poda da arvore de ramificações menos importantes, quanto maior o valor maior a qauntidade de podas e mais simples o modelo se torna

}

In [39]:
modelRFC = RandomForestClassifier(random_state=42)

random_search = RandomizedSearchCV(
    modelRFC,
    param_distributions=param_dist,
    n_iter=250,
    cv=5,
    scoring='accuracy',
    verbose=1,
    random_state=42,
    n_jobs=-1)

random_search.fit(x_train, y_train)

print(f"Melhores parâmetros encontrados: {random_search.best_params_}")
print(f"Melhor score (Accuracy) na validação cruzada: {random_search.best_score_:.4f}")

Fitting 5 folds for each of 250 candidates, totalling 1250 fits
Melhores parâmetros encontrados: {'ccp_alpha': np.float64(0.006887674640884665), 'max_depth': 11, 'max_features': 'sqrt', 'max_samples': 0.6, 'min_samples_leaf': 1, 'min_samples_split': 6, 'n_estimators': 1038}
Melhor score (Accuracy) na validação cruzada: 0.8342


In [40]:
best_model_classifier = random_search.best_estimator_
predictions = best_model_classifier.predict(x_test)

print("--- Métricas de Performance do Modelo RandomForestClassifier ---")
print(f"Acurácia: {accuracy_score(y_test, predictions):.2%}")
print(f"Precisão: {precision_score(y_test, predictions):.2%}")
print(f"Recall: {recall_score(y_test, predictions):.2%}")
print(f"F1-Score: {f1_score(y_test, predictions):.2%}")

# Dica: O classification_report gera um resumo completo de uma vez só
print(classification_report(y_test, predictions))

--- Métricas de Performance do Modelo RandomForestClassifier ---
Acurácia: 81.56%
Precisão: 83.61%
Recall: 68.92%
F1-Score: 75.56%
              precision    recall  f1-score   support

           0       0.81      0.90      0.85       105
           1       0.84      0.69      0.76        74

    accuracy                           0.82       179
   macro avg       0.82      0.80      0.80       179
weighted avg       0.82      0.82      0.81       179



## Aplicando o melhor modelo ao conjunto de teste

In [41]:
previsoes = best_model_classifier.predict(test)

print(previsoes)

[0 0 0 0 0 0 1 0 1 0 0 0 1 0 1 1 0 0 0 1 0 0 1 0 1 0 1 0 0 0 0 0 1 0 0 0 0
 0 0 0 0 0 0 1 1 0 0 0 1 1 0 0 1 1 0 0 0 0 0 1 0 0 0 1 1 1 1 0 0 1 1 0 0 0
 1 0 0 1 0 1 1 0 0 0 0 0 1 0 1 1 0 0 1 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0 0 0 0
 1 1 1 1 0 0 1 0 1 1 0 1 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0
 0 0 1 0 0 1 0 0 1 0 0 1 1 0 1 0 0 0 0 0 1 0 0 0 0 0 0 1 1 0 1 1 0 0 1 0 1
 0 1 0 0 0 0 0 0 0 1 0 1 0 0 0 1 1 0 1 0 0 1 0 1 0 0 0 0 1 0 0 1 0 1 0 1 0
 1 0 1 1 0 1 0 0 0 1 0 0 0 0 0 0 1 1 1 1 0 0 0 0 1 0 1 1 1 0 0 0 0 0 0 0 1
 0 0 0 1 1 0 0 0 0 0 0 0 0 1 1 0 1 0 0 0 0 0 1 1 1 1 0 0 0 0 0 0 1 0 0 0 0
 1 0 0 0 0 0 0 0 1 1 0 1 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 1 0 1 0 0 0 1 0 0
 1 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 1 1 0 0 0 1 0 1 0 0 1 0 1 1 0 1 0 0 0 1 0
 0 1 0 0 1 1 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 1 0 0 0 1 0 1 0 0 1 0 1 0 0 0 0
 0 1 1 1 1 0 0 1 0 0 0]


In [42]:
data_gender["Survived"] = previsoes
data_gender.to_csv("submission.csv", index=False)